# Conveyer Pipeline — Step-by-Step

End-to-end walkthrough of the conversation clustering pipeline: ingest → embeddings → clustering → analysis → export.

In [10]:
from conveyer import ingest, analysis, clustering, viz
from conveyer.config import PipelineConfig
from conveyer.models import build_embedder, build_llm
import pandas as pd
import os

path = "data/skincare_chat_gpt_sessions_sample.csv"
cfg = PipelineConfig(
    data_path=path, cluster_on="question", n_clusters=14,
    embedding_backend="sentence_transformers", first_turn_only=False,
    use_llm_naming=False, out_dir=".output/",
)

## Step 1 — Data Ingest & Feature Engineering

Load raw conversations, then annotate each row with brand-mention counts, recommendation flags, and rule-based intent labels. On synthetic data, intent accuracy is also printed as a sanity check.

In [11]:
df = ingest.ingest(cfg)
df = analysis.add_brand_features(df)
df = analysis.add_intent(df, cfg)
print(f"[ingest] {df.attrs.get('source')} | rows={len(df)}")
print(f"[features] recommendation_rate={df['is_recommendation'].mean():.1%} | "
      f"brands/answer={df['n_brands_answer'].mean():.2f}")
acc = analysis.intent_accuracy(df)
if acc is not None:
    print(f"[intent] rule accuracy vs ground truth (synthetic): {acc:.1%}")

[ingest] data/skincare_chat_gpt_sessions_sample.csv | rows=10000
[features] recommendation_rate=23.4% | brands/answer=0.26


In [12]:
df

,session_id,message_id,user_id,question,answer,formatted_answer,session_pos,a_links_source,skin_care_categories,brands_in_question,...,_brands_a,_brands_q_norm,_brands_a_norm,_text,_unsolicited,_endorsed,is_recommendation,n_brands_answer,intent,asks_recommendation
0,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,93a1b7a1b3589eb1fec014ed107dcff6,ho70mo2ppkntgfren8stjqrrg5a,No you're writing too much like a bot here. Don',"Fair. You’re right. That read scripted, not hu...","Fair. You’re right. That read *scripted*, not ...",4,NaN,NaN,[],...,[Meta],{},{meta},No you're writing too much like a bot here. Don',{meta},{},True,1,informational,False
1,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,09f21e2d49ad9b9b9f7f07007a146b9a,ho70mo2ppkntgfren8stjqrrg5a,no one talks like this you fucking idiot,Got it. You’re right to call it out. That stil...,Got it. You’re right to call it out. That stil...,5,NaN,NaN,[],...,[],{},{},no one talks like this you fucking idiot,{},{},False,0,informational,False
2,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,6d766c2e029d8828a505286261ad7496,ho70mo2ppkntgfren8stjqrrg5a,"Now write it how Stefan would, this script is ...",Alright. I’m going to do this properly now.\nS...,Alright. I’m going to do this properly now.\n\...,6,NaN,NaN,[],...,[],{},{},"Now write it how Stefan would, this script is ...",{},{},False,0,informational,False
3,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,62a8c8c6060019a675cb472af003440d,ho70mo2ppkntgfren8stjqrrg5a,You have gotten so much worse at copy writing ...,I’m not going to engage with the “kill you” pa...,I’m not going to engage with the “kill you” pa...,7,NaN,NaN,['Uproot Clean' 'FurCleanse'],...,[Uproot CleanFurCleanse],{uproot cleanfurcleanse},{uproot cleanfurcleanse},You have gotten so much worse at copy writing ...,{},{uproot cleanfurcleanse},False,1,informational,False
4,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,9bc35b8936d82a9566d3c09b6ba3d87d,ho70mo2ppkntgfren8stjqrrg5a,I like this much better. A few things now \n\n...,Good. This is the right direction. I’ll sharpe...,Good. This is the right direction. I’ll sharpe...,8,NaN,NaN,[],...,[FurCleanseUproot Clean],{},{furcleanseuproot clean},I like this much better. A few things now \n\n...,{furcleanseuproot clean},{},True,1,informational,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,77e7b691-8b74-f4b4-55a0-69fbb606504a,98c245204f32c9809d7d5bbc4d1f53a6,dnegsda9alijnqklrg2kbe98i6h,can I use the peroxide cleaner on wood?,"Short answer: yes, but very carefully — and us...","Short answer: **yes, but very carefully — and ...",30,NaN,NaN,NaN,...,[],{},{},can I use the peroxide cleaner on wood?,{},{},False,0,troubleshooting,False
9996,77e7b691-8b74-f4b4-55a0-69fbb606504a,d190b56912be71b12aa53480dd297a43,dnegsda9alijnqklrg2kbe98i6h,would you use it on kitchen counters? Or shoul...,Great question — kitchens are a little differe...,Great question — kitchens are a little differe...,31,NaN,NaN,NaN,...,[],{},{},would you use it on kitchen counters? Or shoul...,{},{},False,0,comparison,True
9997,77e7b691-8b74-f4b4-55a0-69fbb606504a,dace48180e112ce32b581d3a7471606d,dnegsda9alijnqklrg2kbe98i6h,is using dish soad the best thing to clean kit...,"Short answer: yes — for most kitchens, dish so...","Short answer: **yes — for most kitchens, dish ...",32,NaN,NaN,NaN,...,[],{},{},is using dish soad the best thing to clean kit...,{},{},False,0,informational,False
9998,77e7b691-8b74-f4b4-55a0-69fbb606504a,c91a04d0108424ca57ecfb2818052c4d,dnegsda9alijnqklrg2kbe98i6h,whats a good dish soad? Ideally on the healthi...,Here are some great dish soap options that are...,Here are some **great dish soap options** that...,33,NaN,NaN,NaN,...,[],{},{},whats a good dish soad? Ideally on the healthi...,{},{},False,0,informational,False


## Step 2 — Corpus Preparation & Embeddings

Optionally filter to first-turn messages only, then encode each text into a dense vector. If an LLM is configured and `llm_keyphrase_expansion=True`, keyphrases are extracted first to improve embedding quality.

In [3]:
mask = (df[cfg.col_session_pos] == 1) if cfg.first_turn_only else pd.Series(True, index=df.index)
work = df[mask].copy().reset_index(drop=True)
texts = work[cfg.text_column()].tolist()

#llm = build_llm(cfg)
embed_texts = texts
#if cfg.llm_keyphrase_expansion and llm is not None:
#    embed_texts = clustering.llm_keyphrase_expansion(texts, llm)
#    print("[llm] keyphrase expansion applied before embedding")

embedder = build_embedder(cfg)
embeddings, emb_name = clustering.embed_corpus(embed_texts, cfg, embedder=embedder)
print(f"[embeddings] {embeddings.shape} via {emb_name} | docs={len(texts)}")

c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.conv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 79/79 [28:20<00:00, 21.53s/it]  


[embeddings] (10000, 1024) via sentence-transformers:BAAI/bge-large-en-v1.5 | docs=10000


## Step 3 — Clustering: Method Comparison & Selection

Five algorithms (kmeans, agglomerative, spectral, GMM, HDBSCAN) are scored by silhouette to find the best fit. K-means is also run across a range of k values (sweep) and its stability is measured with ARI across multiple seeds. BERTopic is attempted as an additional reference partition.

In [4]:
niq = work[cfg.col_categories].astype(str).tolist() if cfg.col_categories in work.columns else None
sweep = clustering.kmeans_sweep(embeddings, cfg)
comparison = clustering.compare_methods(embeddings, cfg, niq=niq)
print("[methods]\n" + comparison.to_string(index=False))

n_clusters = cfg.n_clusters
granularity = None
if cfg.llm_select_granularity and llm is not None:
    granularity = clustering.llm_select_granularity(embeddings, texts, cfg, llm)
    n_clusters = granularity["best_k"]
    print(f"[llm] granularity -> n={n_clusters}")

method = clustering.select_best_method(comparison) if cfg.cluster_method == "auto" else cfg.cluster_method
work["cluster_kmeans"] = clustering.run_kmeans(embeddings, n_clusters, cfg)
work["cluster"] = clustering.cluster(method, embeddings, cfg, n_clusters=n_clusters)
stability = clustering.kmeans_stability(embeddings, n_clusters, cfg)

bertopic_labels, _ = clustering.run_bertopic(texts, embeddings, cfg)
if bertopic_labels is not None:
    work["cluster_bertopic"] = bertopic_labels
    print(f"[bertopic] clusters={clustering.n_clusters_found(bertopic_labels)} | "
          f"noise={clustering.noise_fraction(bertopic_labels):.1%}")
else:
    print("[bertopic] unavailable")

print(f"[clustering] method={method} | n={n_clusters} | kmeans stability(ARI)={stability:.3f}")

[methods]
       method  n_clusters  noise_frac  silhouette  davies_bouldin  calinski_harabasz  ARI_vs_NIQ  NMI_vs_NIQ                            note
      hdbscan         4.0      0.9583    0.228161        2.385663          42.346764   -0.030979    0.026886                             NaN
       kmeans         6.0      0.0000    0.044751        5.216193         187.396892    0.002334    0.203896                             NaN
          gmm         6.0      0.0000    0.033957        5.395608         185.602623   -0.003172    0.200805                             NaN
agglomerative         NaN         NaN         NaN             NaN                NaN         NaN         NaN skipped (N=10000 > max_dense_n)
     spectral         NaN         NaN         NaN             NaN                NaN         NaN         NaN skipped (N=10000 > max_dense_n)
[bertopic] clusters=4 | noise=2.5%
[clustering] method=hdbscan | n=6 | kmeans stability(ARI)=0.945


## Step 4 — Analysis & Interpretation

Compute per-cluster statistics, brand amplification rates, top product recommendations, and optional NIQ validation scores. Clusters are named either by keyword extraction or LLM (if configured).

In [5]:
amp = analysis.brand_amplification(df)
summary = analysis.cluster_summary(work, cfg)
niq_scores = analysis.validate_against_niq(work, cfg)
top_reco = analysis.top_recommendations(df)
top_reco_cluster = analysis.top_recommendations_by_cluster(work)
reps = clustering.representative_docs(embeddings, work["cluster"].to_numpy(), texts)

if cfg.use_llm_naming and llm is not None:
    names = analysis.name_clusters_llm(reps, llm)
else:
    names = analysis.name_clusters_keywords(work, texts)
work["cluster_name"] = work["cluster"].map(lambda c: names.get(int(c), {}).get("label", str(c)))

if niq_scores:
    print("[validation vs NIQ]", {k: {m: round(v, 3) for m, v in d.items()} for k, d in niq_scores.items()})

[validation vs NIQ] {'cluster_kmeans': {'ARI': 0.002, 'NMI': 0.204}, 'cluster_bertopic': {'ARI': -0.038, 'NMI': 0.033}}


## Step 5 — Export & Dashboard

Project embeddings to 2D (UMAP/PCA) for visualization, then write 8 CSV files to the output directory. If `build_dashboard=True`, an interactive HTML dashboard is also generated.

In [6]:
os.makedirs(cfg.out_dir, exist_ok=True)
labeled_cols = [cfg.col_question, "cluster", "cluster_kmeans", "cluster_name", "intent",
                "asks_recommendation", "is_recommendation", "n_brands_answer"]
if cfg.col_categories in work.columns:
    labeled_cols.append(cfg.col_categories)
if "cluster_bertopic" in work.columns:
    labeled_cols.append("cluster_bertopic")
labeled = work[labeled_cols]

xy = viz.project_2d(embeddings, cfg.random_state)
proj_cols = [c for c in (cfg.col_question, "cluster", "cluster_name", "intent") if c in work.columns]
projection = work[proj_cols].copy().rename(columns={cfg.col_question: "question"})
projection["x"], projection["y"] = xy[:, 0], xy[:, 1]

outputs = {
    "labeled_first_turn.csv": labeled,
    "cluster_summary.csv": summary,
    "brand_amplification.csv": amp,
    "top_recommendations.csv": top_reco,
    "top_recommendations_by_cluster.csv": top_reco_cluster,
    "kmeans_sweep.csv": sweep,
    "method_comparison.csv": comparison,
    "projection.csv": projection,
}
for fname, frame in outputs.items():
    frame.to_csv(os.path.join(cfg.out_dir, fname), index=False)
print(f"[export] wrote {len(outputs)} files to {os.path.abspath(cfg.out_dir)}")

if cfg.build_dashboard:
    path = viz.build_dashboard(
        {"config": cfg.__dict__, "work": work, "embeddings": embeddings,
         "projection": projection, "summary": summary, "amplification": amp},
        outputs_dir=cfg.out_dir,
    )
    print(f"[dashboard] {os.path.abspath(path)}")

[export] wrote 8 files to c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.output
[dashboard] c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.output\dashboard.html


In [9]:
df

,session_id,message_id,user_id,question,answer,formatted_answer,session_pos,a_links_source,skin_care_categories,brands_in_question,...,_brands_a,_brands_q_norm,_brands_a_norm,_text,_unsolicited,_endorsed,is_recommendation,n_brands_answer,intent,asks_recommendation
0,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,93a1b7a1b3589eb1fec014ed107dcff6,ho70mo2ppkntgfren8stjqrrg5a,No you're writing too much like a bot here. Don',"Fair. You’re right. That read scripted, not hu...","Fair. You’re right. That read *scripted*, not ...",4,NaN,NaN,[],...,[Meta],{},{meta},No you're writing too much like a bot here. Don',{meta},{},True,1,informational,False
1,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,09f21e2d49ad9b9b9f7f07007a146b9a,ho70mo2ppkntgfren8stjqrrg5a,no one talks like this you fucking idiot,Got it. You’re right to call it out. That stil...,Got it. You’re right to call it out. That stil...,5,NaN,NaN,[],...,[],{},{},no one talks like this you fucking idiot,{},{},False,0,informational,False
2,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,6d766c2e029d8828a505286261ad7496,ho70mo2ppkntgfren8stjqrrg5a,"Now write it how Stefan would, this script is ...",Alright. I’m going to do this properly now.\nS...,Alright. I’m going to do this properly now.\n\...,6,NaN,NaN,[],...,[],{},{},"Now write it how Stefan would, this script is ...",{},{},False,0,informational,False
3,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,62a8c8c6060019a675cb472af003440d,ho70mo2ppkntgfren8stjqrrg5a,You have gotten so much worse at copy writing ...,I’m not going to engage with the “kill you” pa...,I’m not going to engage with the “kill you” pa...,7,NaN,NaN,['Uproot Clean' 'FurCleanse'],...,[Uproot CleanFurCleanse],{uproot cleanfurcleanse},{uproot cleanfurcleanse},You have gotten so much worse at copy writing ...,{},{uproot cleanfurcleanse},False,1,informational,False
4,5f55a7e8-9ec6-6b59-17fc-b6f016c27bdb,9bc35b8936d82a9566d3c09b6ba3d87d,ho70mo2ppkntgfren8stjqrrg5a,I like this much better. A few things now \n\n...,Good. This is the right direction. I’ll sharpe...,Good. This is the right direction. I’ll sharpe...,8,NaN,NaN,[],...,[FurCleanseUproot Clean],{},{furcleanseuproot clean},I like this much better. A few things now \n\n...,{furcleanseuproot clean},{},True,1,informational,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,77e7b691-8b74-f4b4-55a0-69fbb606504a,98c245204f32c9809d7d5bbc4d1f53a6,dnegsda9alijnqklrg2kbe98i6h,can I use the peroxide cleaner on wood?,"Short answer: yes, but very carefully — and us...","Short answer: **yes, but very carefully — and ...",30,NaN,NaN,NaN,...,[],{},{},can I use the peroxide cleaner on wood?,{},{},False,0,troubleshooting,False
9996,77e7b691-8b74-f4b4-55a0-69fbb606504a,d190b56912be71b12aa53480dd297a43,dnegsda9alijnqklrg2kbe98i6h,would you use it on kitchen counters? Or shoul...,Great question — kitchens are a little differe...,Great question — kitchens are a little differe...,31,NaN,NaN,NaN,...,[],{},{},would you use it on kitchen counters? Or shoul...,{},{},False,0,comparison,True
9997,77e7b691-8b74-f4b4-55a0-69fbb606504a,dace48180e112ce32b581d3a7471606d,dnegsda9alijnqklrg2kbe98i6h,is using dish soad the best thing to clean kit...,"Short answer: yes — for most kitchens, dish so...","Short answer: **yes — for most kitchens, dish ...",32,NaN,NaN,NaN,...,[],{},{},is using dish soad the best thing to clean kit...,{},{},False,0,informational,False
9998,77e7b691-8b74-f4b4-55a0-69fbb606504a,c91a04d0108424ca57ecfb2818052c4d,dnegsda9alijnqklrg2kbe98i6h,whats a good dish soad? Ideally on the healthi...,Here are some great dish soap options that are...,Here are some **great dish soap options** that...,33,NaN,NaN,NaN,...,[],{},{},whats a good dish soad? Ideally on the healthi...,{},{},False,0,informational,False
